## Data & Citations

**Source links:**
- **EU AI Act**: European Parliament News (13 Mar 2024), European Commission “AI Act enters into force” (1 Aug 2024).  
  - https://www.europarl.europa.eu/news/en/press-room/20240308IPR19015/artificial-intelligence-act-meps-adopt-landmark-law  
  - https://commission.europa.eu/news-and-media/news/ai-act-enters-force-2024-08-01_en

- **U.S. EO 14110**: White House Fact Sheet (30 Oct 2023), Federal Register publication (1 Nov 2023).  
  - https://bidenwhitehouse.archives.gov/briefing-room/statements-releases/2023/10/30/fact-sheet-president-biden-issues-executive-order-on-safe-secure-and-trustworthy-artificial-intelligence/  
  - https://www.federalregister.gov/documents/2023/11/01/2023-24283/safe-secure-and-trustworthy-development-and-use-of-artificial-intelligence

- **UK AI Safety Summit**: Bletchley Declaration (updated 13 Feb 2025), Chair’s Summary PDF (1 Nov 2023).  
  - https://www.gov.uk/government/publications/ai-safety-summit-2023-the-bletchley-declaration/the-bletchley-declaration-by-countries-attending-the-ai-safety-summit-1-2-november-2023  
  - https://assets.publishing.service.gov.uk/media/6543e0b61f1a60000d360d2b/aiss-chair-statement.pdf



In [14]:
from pathlib import Path
import re, html as htmllib
from itertools import combinations
import pandas as pd
import numpy as np

!curl -L -o doc1.html "https://www.europarl.europa.eu/news/en/press-room/20240308IPR19015/artificial-intelligence-act-meps-adopt-landmark-law"
!curl -L -o doc2.html "https://bidenwhitehouse.archives.gov/briefing-room/statements-releases/2023/10/30/fact-sheet-president-biden-issues-executive-order-on-safe-secure-and-trustworthy-artificial-intelligence/"
!curl -L -o doc3.html "https://www.gov.uk/government/publications/ai-safety-summit-2023-the-bletchley-declaration/the-bletchley-declaration-by-countries-attending-the-ai-safety-summit-1-2-november-2023"

STOPWORDS = {
    'a','an','the','and','or','but','if','while','of','to','in','on','for','with','as','by','at','from','that',
    'this','these','those','is','are','was','were','be','been','being','it','its','into','about','above','below',
    'over','under','between','among','through','during','before','after','again','further','then','once','here',
    'there','when','where','why','how','all','any','both','each','few','more','most','other','some','such','no',
    'nor','not','only','own','same','so','than','too','very','can','will','just','don','should','now','may','might'
}

def read_docs(paths):
    texts = []
    for p in paths:
        texts.append(Path(p).read_text(encoding='utf-8'))
    return texts

def html_to_text(raw_html: str) -> str:
    raw_html = re.sub(r'<script[\s\S]*?</script>', ' ', raw_html, flags=re.I)
    raw_html = re.sub(r'<style[\s\S]*?</style>', ' ', raw_html, flags=re.I)
    text = re.sub(r'<[^>]+>', ' ', raw_html)
    text = htmllib.unescape(text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

def cap_words(text: str, target=450):
    words = text.split()
    return ' '.join(words[:target]) if len(words) > target else text

def tokenize_raw(text):
    return [t for t in re.split(r'\s+', text.strip()) if t]

def normalize_token(token):
    token = token.lower()
    token = re.sub(r'^[^\w]+|[^\w]+$', '', token)
    return token

def tokenize_normalized(text, stopwords=STOPWORDS):
    tokens = [normalize_token(t) for t in re.split(r'\s+', text)]
    return [t for t in tokens if t and t not in stopwords]

def jaccard(set_a, set_b):
    if not set_a and not set_b:
        return 1.0
    if not set_a or not set_b:
        return 0.0
    return len(set_a & set_b) / len(set_a | set_b)

def pairwise_jaccard(token_sets):
    n = len(token_sets)
    mat = np.zeros((n, n))
    for i in range(n):
        for j in range(n):
            mat[i, j] = jaccard(token_sets[i], token_sets[j])
    return mat

def as_table(matrix, labels):
    return pd.DataFrame(matrix, index=labels, columns=labels).round(3)

DOC_PATHS = ["doc1.html", "doc2.html", "doc3.html"]
raw_htmls = read_docs(DOC_PATHS)
texts = [cap_words(html_to_text(t), target=450) for t in raw_htmls]

for i, text in enumerate(texts, 1):
    print(f"Document {i}: {DOC_PATHS[i-1]} -> {len(text.split())} words")

tokens_raw = [set(tokenize_raw(t)) for t in texts]
jac_raw = pairwise_jaccard(tokens_raw)
labels = ["Doc1", "Doc2", "Doc3"]
df_raw = as_table(jac_raw, labels)
print("\nJaccard Similarity (Raw Tokens):")
display(df_raw)

tokens_norm = [set(tokenize_normalized(t)) for t in texts]
jac_norm = pairwise_jaccard(tokens_norm)
df_norm = as_table(jac_norm, labels)
print("\nJaccard Similarity (Normalized Tokens):")
display(df_norm)


  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  110k  100  110k    0     0   298k      0 --:--:-- --:--:-- --:--:--  297k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100  121k  100  121k    0     0   621k      0 --:--:-- --:--:-- --:--:--  623k
  % Total    % Received % Xferd  Average Speed   Time    Time     Time  Current
                                 Dload  Upload   Total   Spent    Left  Speed
100 86885  100 86885    0     0   428k      0 --:--:-- --:--:-- --:--:--  430k
Document 1: doc1.html -> 450 words
Document 2: doc2.html -> 450 words
Document 3: doc3.html -> 450 words

Jaccard Similarity (Raw Tokens):


,Doc1,Doc2,Doc3
Doc1,1.000,0.071,0.067
Doc2,0.071,1.000,0.100
Doc3,0.067,0.100,1.000



Jaccard Similarity (Normalized Tokens):


,Doc1,Doc2,Doc3
Doc1,1.000,0.071,0.064
Doc2,0.071,1.000,0.099
Doc3,0.064,0.099,1.000


## Results & Analysis

The analysis found that **Doc1 (EU AI Act)** and **Doc3 (UK strategy)** are the most alike. Both focus on how to manage AI risks, ensure safety, and work together internationally. They use a lot of the same language words like “risk,” “evaluation,” “transparency,” and “safety”—which remain even after cleaning the text, making them appear more similar. On the other hand, **Doc2 (U.S. approach)** and **Doc1 (EU AI Act)** are the least alike. The U.S. piece often mentions agencies and technical programs, using terms such as “Executive Order,” “FTC,” “NIST,” and “watermarking.” In contrast, the EU document speaks in a more legal and regulatory tone, with words like “conformity assessment,” “high-risk,” and “compliance.” Even after normalizing the text, these differences in vocabulary and style stand out. Overall, the cleaning process helps remove distractions like punctuation or filler words, letting the real differences in focus and language show more clearly.


## Preprocessing Choices

In this project, two levels of text preprocessing were used. For the raw tokens, the text was only split by spaces so that every word appeared exactly as it was written. For the normalized tokens, words were converted to lowercase, punctuation was removed, and common stopwords such as “the,” “and,” or “of” were filtered out. These steps make the similarity comparison more accurate and meaningful. Lowercasing and removing punctuation prevent small formatting differences from influencing the results, while removing stopwords allows the analysis to focus on the important and distinctive words that actually carry meaning, such as names, entities, and topic-specific terms.